<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class Step:
    action: Any
    prob: float  # New: the probability that we ended up here
    reward: float
    state: Any
class MDP:
    def start_state(self) -> Any:
        raise NotImplementedError

    def successors(self, state: Any) -> list[Step]:
        raise NotImplementedError

    def is_end(self, state: Any) -> bool:
        raise NotImplementedError
    def discount(self) -> float:
        raise NotImplementedError
class FlakyTramMDP(MDP):
    def __init__(self, num_locs: int, failure_prob: float):
        self.num_locs = num_locs
        self.failure_prob = failure_prob
    def start_state(self) -> Any:
        return 1

    def successors(self, state: Any) -> list[Step]:
        successors = []
        # Walk
        if state + 1 <= self.num_locs:
            successors.append(Step(action="walk", prob=1, reward=-1, state=state + 1))
        # Tram
        if 2 * state <= self.num_locs:
            # Success: move to desired state
            successors.append(Step(action="tram", prob=1 - self.failure_prob, reward=-2, state=2 * state))
            # Failure: stay in the same state
            successors.append(Step(action="tram", prob=self.failure_prob, reward=-2, state=state))
        return successors
    def is_end(self, state: Any) -> bool:
        return state == self.num_locs
    def discount(self) -> float:
        # No discounting for now
        return 1


In [23]:
mdp = FlakyTramMDP(num_locs=10, failure_prob=0.4)
state = mdp.start_state()
successors = mdp.successors(state)
is_end = mdp.is_end(successors[0].state)

In [24]:
print(f"Is end state: {is_end}")
print(f"Current state: {state}")
print(f"Successors: {successors}")

Is end state: False
Current state: 1
Successors: [Step(action='walk', prob=1, reward=-1, state=2), Step(action='tram', prob=0.6, reward=-2, state=2), Step(action='tram', prob=0.4, reward=-2, state=1)]


### Dice Example

In [25]:
class DiceGameMDP(MDP):
    def start_state(self) -> Any:
        return "in"

    def successors(self, state: Any) -> list[Step]:
        return [
            Step(action="quit", prob=1, reward=10, state="end"),
            Step(action="stay", prob=1/3, reward=4, state="end"),
            Step(action="stay", prob=2/3, reward=4, state="in"),
        ]

    def is_end(self, state: Any) -> bool:
        return state == "end"

    def discount(self) -> float:
        return 1

In [26]:
mdp = DiceGameMDP()
state = mdp.start_state()
successors = mdp.successors(state)
is_end = mdp.is_end(successors[0].state)

In [27]:
print(f"Is end state: {is_end}")
print(f"Current state: {state}")
print(f"Successors: {successors}")

Is end state: True
Current state: in
Successors: [Step(action='quit', prob=1, reward=10, state='end'), Step(action='stay', prob=0.3333333333333333, reward=4, state='end'), Step(action='stay', prob=0.6666666666666666, reward=4, state='in')]


### Policy

In [28]:
def always_stay_policy(state: int) -> str:
    return "stay"
def always_quit_policy(state: int) -> str:
    return "quit"
def always_walk_policy(state: int) -> str:
    return "walk"
def tram_if_possible_policy(mdp: MDP, state: int) -> str:
    """Need the MDP to know the number of locations to make sure we can take the tram."""
    if state * 2 <= mdp.num_locs:
        return "tram"
    else:
        return "walk"

In [29]:
mdp = FlakyTramMDP(num_locs=10, failure_prob=0.4)
action = always_walk_policy(state=3)
print(f"Action: {action}")

action = tram_if_possible_policy(mdp, state=3)
print(f"Action: {action}")

action = tram_if_possible_policy(mdp, state=6)
print(f"Action: {action}")

Action: walk
Action: tram
Action: walk


###Rollout

In [34]:
from typing import Callable
import numpy as np

Policy = Callable[[Any], Any]

@dataclass
class Rollout:
    """Represents a rollout of an MDP (sequence of actions that produces a utility)."""
    steps: list[Step]
    discount: float
    utility: float  # Discounted sum of rewards
    def __init__(self, steps: list[Step], discount: float):
        self.steps = steps
        self.discount = discount
        self.utility = compute_utility(steps, discount)

def compute_utility(steps: list[Step], discount: float) -> float:
    """Computes the utility (discounted sum of rewards) of a rollout."""
    rewards = [step.reward * discount ** i for i, step in enumerate(steps)]
    utility = sum(rewards)
    return utility

def generate_rollout(mdp: MDP, policy: Policy) -> Rollout:
    """Run the `policy` in the `mdp` and return the rollout."""
    steps = []
    state = mdp.start_state()
    while not mdp.is_end(state):
        # Policy: choose an action
        action = policy(state)
        # MDP: choose a successor according to that action
        successors = [successor for successor in mdp.successors(state) if successor.action == action]
        probs = [successor.prob for successor in successors]
        choice = np.random.choice(len(successors), p=probs)
        step = successors[choice]
        steps.append(step)
        # Advance to the next state
        state = step.state
    return Rollout(steps=steps, discount=mdp.discount())

In [35]:
np.random.seed(1)
rollout = generate_rollout(mdp, always_walk_policy)

In [36]:
print(rollout)

Rollout(steps=[Step(action='walk', prob=1, reward=-1, state=2), Step(action='walk', prob=1, reward=-1, state=3), Step(action='walk', prob=1, reward=-1, state=4), Step(action='walk', prob=1, reward=-1, state=5), Step(action='walk', prob=1, reward=-1, state=6), Step(action='walk', prob=1, reward=-1, state=7), Step(action='walk', prob=1, reward=-1, state=8), Step(action='walk', prob=1, reward=-1, state=9), Step(action='walk', prob=1, reward=-1, state=10)], discount=1, utility=-9)
